In [28]:
from langchain_community.document_loaders import WebBaseLoader

urls = [
    "https://lilianweng.github.io/posts/2024-11-28-reward-hacking/",
    "https://lilianweng.github.io/posts/2024-07-07-hallucination/",
    "https://lilianweng.github.io/posts/2024-04-12-diffusion-video/",
]

docs = [WebBaseLoader(url).load() for url in urls]

In [29]:
from dotenv import load_dotenv
load_dotenv()

import os
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")

In [2]:
docs

[[Document(metadata={'source': 'https://lilianweng.github.io/posts/2024-11-28-reward-hacking/', 'title': "Reward Hacking in Reinforcement Learning | Lil'Log", 'description': 'Reward hacking occurs when a reinforcement learning (RL) agent exploits flaws or ambiguities in the reward function to achieve high rewards, without genuinely learning or completing the intended task. Reward hacking exists because RL environments are often imperfect, and it is fundamentally challenging to accurately specify a reward function.\nWith the rise of language models generalizing to a broad spectrum of tasks and RLHF becomes a de facto method for alignment training, reward hacking in RL training of language models has become a critical practical challenge. Instances where the model learns to modify unit tests to pass coding tasks, or where responses contain biases that mimic a user’s preference, are pretty concerning and are likely one of the major blockers for real-world deployment of more autonomous use

# Split the fetched document into smaller chunks for indexing into our vectorstore

In [30]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

docs_list = [item for sublist in docs for item in sublist]
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=100, length_function=len
)

In [31]:
docs_split = text_splitter.split_documents(docs_list)
docs_split

[Document(metadata={'source': 'https://lilianweng.github.io/posts/2024-11-28-reward-hacking/', 'title': "Reward Hacking in Reinforcement Learning | Lil'Log", 'description': 'Reward hacking occurs when a reinforcement learning (RL) agent exploits flaws or ambiguities in the reward function to achieve high rewards, without genuinely learning or completing the intended task. Reward hacking exists because RL environments are often imperfect, and it is fundamentally challenging to accurately specify a reward function.\nWith the rise of language models generalizing to a broad spectrum of tasks and RLHF becomes a de facto method for alignment training, reward hacking in RL training of language models has become a critical practical challenge. Instances where the model learns to modify unit tests to pass coding tasks, or where responses contain biases that mimic a user’s preference, are pretty concerning and are likely one of the major blockers for real-world deployment of more autonomous use 

In [32]:
docs_split[0].page_content.strip()

"Reward Hacking in Reinforcement Learning | Lil'Log\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nLil'Log\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n|\n\n\n\n\n\n\nPosts\n\n\n\n\nArchive\n\n\n\n\nSearch\n\n\n\n\nTags\n\n\n\n\nFAQ\n\n\n\n\n\n\n\n\n\n      Reward Hacking in Reinforcement Learning\n    \nDate: November 28, 2024  |  Estimated Reading Time: 37 min  |  Author: Lilian Weng\n\n\n \n\n\nTable of Contents\n\n\n\nBackground\n\nReward Function in RL\n\nSpurious Correlation\n\n\nLet’s Define Reward Hacking\n\nList of Examples\n\nReward hacking examples in RL tasks"

# Use in memory vector store and store openai embeddings  

In [33]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

vectorstore = InMemoryVectorStore.from_documents(documents=docs_split, embedding=OpenAIEmbeddings(api_key=OPENAI_API_KEY))

In [34]:
retriever = vectorstore.as_retriever()

In [35]:
from langchain.tools.retriever import create_retriever_tool
retriever_tool = create_retriever_tool(retriever, "retrieve_blog_posts", "Search and return information about Lilian Weng blog posts.")

In [36]:
retriever_tool.invoke({"query": "types of reward hacking"})

'Detecting Reward Hacking#\n\nIn-Context Reward Hacking#\n\n(Note: Some work defines reward tampering as a distinct category of misalignment behavior from reward hacking. But I consider reward hacking as a broader concept here.)\nAt a high level, reward hacking can be categorized into two types: environment or goal misspecification, and reward tampering.\n\nPan et al. (2022) investigated reward hacking as a function of agent capabilities, including (1) model size, (2) action space resolution, (3) observation space noise, and (4) training time. They also proposed a taxonomy of three types of misspecified proxy rewards:'

In [37]:
from langchain_openai import ChatOpenAI
from langgraph.graph import MessagesState
llm = ChatOpenAI(model="gpt-4o", api_key=OPENAI_API_KEY)


In [38]:
def generate_query_or_response(state: MessagesState):
    """Generate a query or response based on the last message in the state.
    If the last message is from the user, return a query prompt.
    """

    # either we do retrirval or rewrite the query - this function will decide whether to retrieve or generate a response
    llm_with_retriever = llm.bind_tools([retriever_tool])
    response = llm_with_retriever.invoke(
      state["messages"]
    )
    print(response,"response from llm")
    return {"messages" : [response]}

In [39]:
input = {"messages": [{"role": "user", "content": "hello!"}]}
generate_query_or_response(input)["messages"][-1].pretty_print()

content='Hello! How can I assist you today?' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 61, 'total_tokens': 71, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_07871e2ad8', 'id': 'chatcmpl-C12YY5WfzpVCwHWiKf43PgRFC2hvy', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='run--a19dac75-f399-474f-a84e-6d661bd38218-0' usage_metadata={'input_tokens': 61, 'output_tokens': 10, 'total_tokens': 71, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}} response from llm
================================== Ai Message ==================================

Hello! How can I assist you today?


In [13]:
input = {
    "messages": [
        {
            "role": "user",
            "content": "What does Lilian Weng say about types of reward hacking?",
        }
    ]
}
generate_query_or_response(input)["messages"][-1].pretty_print()

content='' additional_kwargs={'tool_calls': [{'id': 'call_SqsgPuShUq4bCckEIH7Tc3Qg', 'function': {'arguments': '{"query":"types of reward hacking"}', 'name': 'retrieve_blog_posts'}, 'type': 'function'}], 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 72, 'total_tokens': 90, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_07871e2ad8', 'id': 'chatcmpl-C11i40hhtLb2TBxdhB42yb074gODO', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='run--d8ee5338-6c2a-449e-af22-0319cf69e067-0' tool_calls=[{'name': 'retrieve_blog_posts', 'args': {'query': 'types of reward hacking'}, 'id': 'call_SqsgPuShUq4bCckEIH7Tc3Qg', 'type': 'tool_call'}] usage_metadata={'input_tokens': 72, 'output_tokens': 18, 'total_tokens': 9

# Grade Document 

TO check if the retrieved document is relevant to the question or not 

In [15]:
# Upon retriveing the documents, we need ti have grader tool which will checks the documents which is retruved and then it will return the noide called generate the response
# or rewrite the query and then passes to llm for retrival tool call

In [40]:
from pydantic import BaseModel, Field
from typing import Literal

GRADE_PROMPT = (
    "You are a grader assessing relevance of a retrieved document to a user question. \n "
    "Here is the retrieved document: \n\n {context} \n\n"
    "Here is the user question: {question} \n"
    "If the document contains keyword(s) or semantic meaning related to the user question, grade it as relevant. \n"
    "Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the question."
)

In [41]:
class GradeDocuments(BaseModel):

    binary_score: str = Field(
        description="Relevance score: 'yes' if relevant, or 'no' if not relevant"
    )
# Response format from grader tool 


In [42]:
grader_model = ChatOpenAI(model="gpt-4o", api_key=OPENAI_API_KEY)

In [ ]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, ToolMessage, BaseMessage
from typing import List


def grade_documents(state: MessagesState):
    """Grade the retrieved documents based on their relevance to the user question."""

    messages: List[BaseMessage] = state["messages"]

    # Step 1: Extract user query (first HumanMessage)
    query = next((m.content for m in messages if isinstance(m, HumanMessage)), None)

    # Step 2: Extract the last ToolMessage content (retrieved docs)
    retrieved_docs = next((m.content for m in reversed(messages) if isinstance(m, ToolMessage)), None)


    print(query, "query from user")
    print(retrieved_docs, "retrieved documents")

    # Step 1: Create the parser
    parser = JsonOutputParser(pydantic_object=GradeDocuments)

    # Step 2: Create the prompt template using parser format instructions
    prompt = PromptTemplate(
        template=GRADE_PROMPT,
        input_variables=["context", "question"],
        partial_variables={"format_instructions": parser.get_format_instructions()},
    )

    # Step 3: Format the prompt with actual values
    formatted_prompt = prompt.format(
        context=retrieved_docs,
        question=query
    )

    print(formatted_prompt, "Formatted Grader Prompt")

    # Step 4: Invoke the grader model with the formatted prompt
    response = grader_model.invoke(formatted_prompt)
    print(response, "Grader Response")

    # Step 5: Parse the response with the JsonOutputParser
    parsed_result = response.content
    print(parsed_result, "Parsed Grader Response")

    # Example: Assuming parsed_result has a field `score` or `relevant` as a boolean or string
    if parsed_result.lower() == "yes":
        return "generate_answer"
    elif parsed_result.lower() == "no":
        return "rewrite_query"


In [62]:
from langchain_core.messages import convert_to_messages

input = {
    "messages": convert_to_messages([
        {"role": "user", "content": "What does Lilian Weng say about types of reward hacking?"},
       {
                "role": "assistant",
                "content": "",
                "tool_calls": [
                    {
                        "id": "1",
                        "name": "retrieve_blog_posts",
                        "args": {"query": "types of reward hacking"},
                    }
                ],
            },
        {"role": "tool", "content": "meow", "tool_call_id": "1"},
    ])
}

# we are passing lits of conversations i.e user questiion, tool call from llm and tool response
response = grade_documents(input)

print(response, "Final Response from Grader Tool")

What does Lilian Weng say about types of reward hacking? query from user
meow retrieved documents
You are a grader assessing relevance of a retrieved document to a user question. 
 Here is the retrieved document: 

 meow 

Here is the user question: What does Lilian Weng say about types of reward hacking? 
If the document contains keyword(s) or semantic meaning related to the user question, grade it as relevant. 
Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the question. Formatted Grader Prompt
content='No' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 1, 'prompt_tokens': 98, 'total_tokens': 99, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_07871e2ad8', 'id': 'chatcmpl-C12qtbd9pHsk

In [64]:
input = {
    "messages": convert_to_messages(
        [
            {
                "role": "user",
                "content": "What does Lilian Weng say about types of reward hacking?",
            },
            {
                "role": "assistant",
                "content": "",
                "tool_calls": [
                    {
                        "id": "1",
                        "name": "retrieve_blog_posts",
                        "args": {"query": "types of reward hacking"},
                    }
                ],
            },
            {
                "role": "tool",
                "content": "reward hacking can be categorized into two types: environment or goal misspecification, and reward tampering.\n\nPan et al. (2022) investigated reward hacking as a function of agent capabilities, including (1) model size, (2) action space resolution,",
                "tool_call_id": "1",
            },
        ]
    )
}
grade_documents(input)

What does Lilian Weng say about types of reward hacking? query from user
reward hacking can be categorized into two types: environment or goal misspecification, and reward tampering.

Pan et al. (2022) investigated reward hacking as a function of agent capabilities, including (1) model size, (2) action space resolution, retrieved documents
You are a grader assessing relevance of a retrieved document to a user question. 
 Here is the retrieved document: 

 reward hacking can be categorized into two types: environment or goal misspecification, and reward tampering.

Pan et al. (2022) investigated reward hacking as a function of agent capabilities, including (1) model size, (2) action space resolution, 

Here is the user question: What does Lilian Weng say about types of reward hacking? 
If the document contains keyword(s) or semantic meaning related to the user question, grade it as relevant. 
Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the que

'rewrite_query'